# Hill method, parity-protected cat (corrected 12/10/2026)

Rewrite of `Hill_method.ipynb` (this folder) with the corrections of `studies/hill_orbits/Hill_method_corrected.ipynb`:

1. **Nonlinear force.** Derived from the Julia EOM `equation_parity_cos_sin!` (flux form), i.e. from
   $H = \dots - 2E_J\sin\varepsilon_p[\cos(\sqrt2\varphi_a x_a)\sin(\sqrt2\varphi_b x_b) - \sqrt2\varphi_b x_b]$:
   $\mathbf f_{nl} = \sqrt2\,\bigl(\beta_a'\sin(\sqrt2 x_a)\sin(\sqrt2 x_b),\; -\beta_b'[\cos(\sqrt2 x_a)\cos(\sqrt2 x_b)-1]\bigr)^T$.
   The former notebook had the opposite sign on the memory force and no $\sqrt2$ prefactor on both forces.
2. **Floquet exponents.** The 2n genuine exponents are selected by the centring of their eigenvector (one member per family),
   with the Liouville check, as in `Hill_method_corrected.ipynb`.
3. **Damping and parameters.** Julia convention $\ddot\phi_b = \dots - 2\kappa_b\dot\phi_b$, so $C = \mathrm{diag}(0, \kappa_b')$,
   with $\kappa_b = \pi/10.4$ (was $1/10.4$ and $\kappa_b'/2$).
4. **$g_2$.** $g_2 = E_J\sin(\varepsilon_p)\varphi_a^2\varphi_b$ (static pump), $\varepsilon_d = 10.12\,g_2$ at every $\varepsilon_p$
   (the former sweep used the $g_2$ of the previous step, with $J_1$).
5. **Seed.** The satellite is found by integrating the EOMs in time from a grid of initial conditions, at $g_2/2\pi = 100$ MHz,
   and projecting the last period on the Fourier basis. The harmonic balance then refines it.
6. **AFT grid.** One period sampled uniformly (`endpoint=False`), so the transform is exact for the retained harmonics.

Outputs: `data/variants/floquet_parity_protected_12_10_2026.json` and
`figures/variants/floquet_multiplier_parity_protected_12_10_2026.pdf`.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import block_diag, eig as scipy_eig
from scipy.optimize import root, linear_sum_assignment
from scipy.integrate import solve_ivp
import warnings
warnings.filterwarnings("ignore", message=".*encountered in matmul")   # spurious with numpy 2 + Accelerate on macOS
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import *

In [ ]:
w_a = 25.338776456203686
w_b = 2*w_a
phi_a = 0.11
phi_b = 0.204
E_J = 37.12*2*np.pi*1.5
kappa_b = np.pi/10.4   # 1/ns, Julia convention (phi_b'' = ... - 2 kappa_b phi_b')
eps_d_over_g2 = 10.12  # as in Hill_method_corrected.ipynb (Fig. 4)

def g2_of(eps_p):
    """Two-photon rate for the static pump (w_p = 0): g2 = E_J sin(eps_p) phi_a^2 phi_b."""
    return np.sin(eps_p)*E_J*phi_a**2*phi_b

def eps_p_of(g2_MHz):
    return np.arcsin(g2_MHz*2*np.pi*1e-3/(E_J*phi_a**2*phi_b))

def set_params(eps_p):
    """Primed parameters, read as globals by f_nl_time and db_dz."""
    global beta_a_p, beta_b_p, eps_d_p
    beta_a_p = 2*E_J*np.sin(eps_p)*phi_a**2/w_a
    beta_b_p = 2*E_J*np.sin(eps_p)*phi_b**2/w_b
    eps_d_p = eps_d_over_g2*g2_of(eps_p)*phi_b/w_b

n = 2                       # number of modes
omega, nu = 2.0, 2          # drive at harmonic 2 of tau = w_a t, period 2 pi
M = np.array([[1.0, 0.0], [0.0, 0.25]])
C = np.array([[0.0, 0.0], [0.0, kappa_b/w_b]])
K = np.eye(2)

In [ ]:
def build_L(M, C, K, N_H, omega=1.0, nu=1.0):
    """Linear HB operator for the ordering [c0, s1, c1, ..., s_NH, c_NH], each block in R^n."""
    n = M.shape[0]
    L = np.zeros((n*(2*N_H + 1), n*(2*N_H + 1)))
    L[:n, :n] = K
    for k in range(1, N_H+1):
        A = K - ((k*omega/nu)**2)*M
        B = (k*omega/nu)*C
        is_, ic_ = 1 + 2*(k-1), 2 + 2*(k-1)
        L[is_*n:(is_+1)*n, is_*n:(is_+1)*n] = A
        L[is_*n:(is_+1)*n, ic_*n:(ic_+1)*n] = -B
        L[ic_*n:(ic_+1)*n, is_*n:(is_+1)*n] = B
        L[ic_*n:(ic_+1)*n, ic_*n:(ic_+1)*n] = A
    return L

def build_b_ext(N_H, eps_d_p, nu=2):
    """Forcing f_ext = -sqrt(2) eps_d' cos(2 tau) e_b, in the ordering of z."""
    b = np.zeros(n*(2*N_H + 1))
    ic2 = 2 + 2*(nu-1)
    b[ic2*n:(ic2+1)*n] = np.array([0.0, -np.sqrt(2.0)*eps_d_p])
    return b

def build_Gamma(t, N_H, n, omega=2.0, nu=2.0):
    """Gamma = Phi ⊗ I_n (time-major stacking), Phi columns [1/sqrt2, sin(kt), cos(kt), ...]."""
    k = np.arange(1, N_H + 1)
    Phi = np.empty((len(t), 2*N_H + 1))
    Phi[:, 0] = 1.0/np.sqrt(2.0)
    Phi[:, 1::2] = np.sin(np.outer(t, k*omega/nu))
    Phi[:, 2::2] = np.cos(np.outer(t, k*omega/nu))
    return np.kron(Phi, np.eye(n))

def x_tilde_to_X(tildex, n, N):
    return tildex.reshape(N, n).T

def X_to_x_tilde(X):
    return X.T.reshape(-1)

def f_nl_time(X):
    """Parity-protected nonlinearity, X of shape (2, N): f = sqrt2 (b_a sin sin, -b_b (cos cos - 1))."""
    s2 = np.sqrt(2.0)
    sa, ca = np.sin(s2*X[0]), np.cos(s2*X[0])
    sb, cb = np.sin(s2*X[1]), np.cos(s2*X[1])
    return np.array([s2*beta_a_p*sa*sb, -s2*beta_b_p*(ca*cb - 1.0)])

def b_nl(z, Gamma, Gamma_pinv, n, N):
    X = x_tilde_to_X(Gamma @ z, n, N)
    return Gamma_pinv @ X_to_x_tilde(f_nl_time(X))

def residual(z, L, b_ext, Gamma, Gamma_pinv, n, N):
    return L @ z + b_nl(z, Gamma, Gamma_pinv, n, N) - b_ext

def db_dz(z, Gamma, Gamma_pinv, n, N):
    """Chain rule Gamma^+ (df/dX) Gamma, with df/dX = 2 [[b_a ca sb, b_a sa cb], [b_b sa cb, b_b ca sb]] at each sample."""
    X = x_tilde_to_X(Gamma @ z, n, N)
    s2 = np.sqrt(2.0)
    sa, ca = np.sin(s2*X[0]), np.cos(s2*X[0])
    sb, cb = np.sin(s2*X[1]), np.cos(s2*X[1])
    J = 2.0*np.array([[beta_a_p*ca*sb, beta_a_p*sa*cb],
                      [beta_b_p*sa*cb, beta_b_p*ca*sb]])        # (2, 2, N)
    G = Gamma.reshape(N, n, -1)
    JG = np.einsum('abj,jbm->jam', J, G).reshape(n*N, -1)
    return Gamma_pinv @ JG

def jacobian(z, L, Gamma, Gamma_pinv, n, N):
    return L + db_dz(z, Gamma, Gamma_pinv, n, N)

def powell_hybrid_solve(z0, L, b_ext, Gamma, Gamma_pinv, n, N, tol=1e-10, maxit=50, verbose=True):
    sol = root(lambda z: residual(z, L, b_ext, Gamma, Gamma_pinv, n, N), z0,
               jac=lambda z: jacobian(z, L, Gamma, Gamma_pinv, n, N),
               method="hybr", tol=tol, options={"maxfev": maxit*(len(z0) + 1)})
    if not sol.success:
        raise RuntimeError(f"Powell hybrid did not converge: {sol.message}")
    if verbose: print("Converged", sol.nfev, np.linalg.norm(sol.fun))
    return sol.x

def setup_HB(N_H, n_samples=64):
    """Globals used by Floq_coef: Gamma, Gamma_inverse, N (one period, uniform, endpoint excluded)."""
    global Gamma, Gamma_inverse, N
    N = n_samples*N_H
    tau = np.linspace(0, 2*np.pi, N, endpoint=False)
    Gamma = build_Gamma(tau, N_H, n, omega, nu)
    Gamma_inverse = np.linalg.pinv(Gamma)
    return build_L(M, C, K, N_H, omega, nu)

def check_L_vs_time(N_H, trials=3):
    """L z against M x'' + C x' + K x computed in time, for random z (same check as check_L_vs_Gamma)."""
    t = np.linspace(0, 2*np.pi, 16*N_H, endpoint=False)
    k = np.arange(1, N_H + 1)
    Phi, dPhi, ddPhi = (np.zeros((len(t), 2*N_H + 1)) for _ in range(3))
    Phi[:, 0] = 1/np.sqrt(2)
    Phi[:, 1::2], Phi[:, 2::2] = np.sin(np.outer(t, k)), np.cos(np.outer(t, k))
    dPhi[:, 1::2], dPhi[:, 2::2] = k*np.cos(np.outer(t, k)), -k*np.sin(np.outer(t, k))
    ddPhi[:, 1::2], ddPhi[:, 2::2] = -k**2*np.sin(np.outer(t, k)), -k**2*np.cos(np.outer(t, k))
    L = build_L(M, C, K, N_H, omega, nu)
    for _ in range(trials):
        z = np.random.randn(n*(2*N_H + 1))
        x, dx, ddx = (np.kron(P, np.eye(n)) @ z for P in (Phi, dPhi, ddPhi))
        lin_time = np.kron(np.eye(len(t)), M) @ ddx + np.kron(np.eye(len(t)), C) @ dx + np.kron(np.eye(len(t)), K) @ x
        err = np.linalg.norm(lin_time - np.kron(Phi, np.eye(n)) @ (L @ z))/np.linalg.norm(lin_time)
        print(f"relative error = {err:.1e}")

check_L_vs_time(5)

In [ ]:
# Floquet exponents: same functions as Hill_method_corrected.ipynb
def complex_harmonics(U, N_H, n):
    A = np.zeros((2*N_H + 1, n, U.shape[1]), dtype=complex)
    A[N_H] = U[:n, :]/np.sqrt(2)
    for k in range(1, N_H + 1):
        s = U[(2*k - 1)*n:(2*k)*n, :]
        c = U[(2*k)*n:(2*k + 1)*n, :]
        A[N_H + k] = (c - 1j*s)/2
        A[N_H - k] = (c + 1j*s)/2
    return A

def harmonic_centroid(A, N_H):
    w = np.sum(np.abs(A)**2, axis=1)
    k = np.arange(-N_H, N_H + 1)[:, None]
    return np.sum(k*w, axis=0)/np.sum(w, axis=0)

def shifted_overlap(a, b, m, N_H):
    if m > 0:
        a_s, b_s = a[m:], b[:-m]
    elif m < 0:
        a_s, b_s = a[:m], b[-m:]
    else:
        a_s, b_s = a, b
    return np.abs(np.vdot(a_s, b_s))/(np.linalg.norm(a)*np.linalg.norm(b) + 1e-300)

def select_genuine(eigvals, eigvecs, N_H, n, tol=1e-3, overlap_min=0.9):
    """Keep the 2n genuine exponents (one per family lambda + i m), folded into |Im| <= 1/2."""
    U = eigvecs[eigvecs.shape[0]//2:, :]
    A = complex_harmonics(U, N_H, n)
    centroid = harmonic_centroid(A, N_H)
    order = np.lexsort((np.abs(np.imag(eigvals)), np.abs(centroid)))
    chosen = []
    for j in order:
        lam = eigvals[j]
        if not np.isfinite(lam):
            continue
        same_family = False
        for c in chosen:
            d = lam - eigvals[c]
            m = int(np.round(np.imag(d)))
            if (m != 0 and abs(np.real(d)) < tol and abs(np.imag(d) - m) < tol
                    and shifted_overlap(A[:, :, c], A[:, :, j], m, N_H) > overlap_min):
                same_family = True
                break
        if not same_family:
            chosen.append(j)
        if len(chosen) == 2*n:
            break
    chosen = np.array(chosen)
    lam = eigvals[chosen]
    lam_folded = lam - 1j*np.round(np.imag(lam))
    lam_folded = np.where(np.abs(np.imag(lam_folded) + 0.5) < 1e-6, lam_folded + 1j, lam_folded)
    return lam_folded, eigvecs[:, chosen]

def liouville_residual(eigvals, M, C):
    """Sum of Re(lambda) + tr(M^-1 C), should be ~0."""
    return np.sum(np.real(eigvals)) + np.trace(np.linalg.solve(M, C))

def Floq_coef(z_sol, L, M, C, N_H, n):
    dim = (2*N_H + 1)*n
    Delta_1 = np.zeros([dim, dim])
    for i in range(2*N_H + 1):
        Delta_1[i*n:(i+1)*n, i*n:(i+1)*n] = C
    for k in range(1, N_H + 1):
        i, j = 2*k - 1, 2*k      # sin, cos indices
        Delta_1[i*n:(i+1)*n, j*n:(j+1)*n] = -2*k*M
        Delta_1[j*n:(j+1)*n, i*n:(i+1)*n] = 2*k*M
    Delta_2 = block_diag(*([M]*(2*N_H + 1)))
    h_z = jacobian(z_sol, L, Gamma, Gamma_inverse, n, N)
    B_1 = np.block([[Delta_1, h_z], [-np.eye(dim), np.zeros([dim, dim])]])
    B_2 = -np.block([[Delta_2, np.zeros([dim, dim])], [np.zeros([dim, dim]), np.eye(dim)]])
    eigvals, eigvecs = scipy_eig(B_1, B_2)
    return select_genuine(eigvals, eigvecs, N_H, n)

def match_eigs(prev_eigvals, prev_eigvecs, new_eigvals, new_eigvecs, alpha=1.0, beta=5.0, T=2*np.pi, n_dof=2):
    """Continuation sorting: multiplier distance + eigenvector overlap (maximised over harmonic shifts)."""
    m = len(prev_eigvals)
    cost = np.zeros((m, m))
    n_rows = prev_eigvecs.shape[0]//2
    N_H_v = (n_rows//n_dof - 1)//2
    A_prev = complex_harmonics(prev_eigvecs[n_rows:, :], N_H_v, n_dof)
    A_new = complex_harmonics(new_eigvecs[n_rows:, :], N_H_v, n_dof)
    mu_prev, mu_new = np.exp(prev_eigvals*T), np.exp(new_eigvals*T)
    mu_scale = np.max(np.abs(mu_prev)) + 1e-12
    for i in range(m):
        for j in range(m):
            overlap = max(shifted_overlap(A_prev[:, :, i], A_new[:, :, j], s, N_H_v) for s in (-1, 0, 1))
            cost[i, j] = alpha*np.abs(mu_prev[i] - mu_new[j])/mu_scale + beta*(1.0 - overlap)
    return linear_sum_assignment(cost)[1]

In [ ]:
def orbit_complex(z, N_H, t=np.linspace(0, 2*np.pi, 400)):
    """Rotating-frame orbit in units of the ladder amplitudes a and b (phi_a (a + a^*) = sqrt2 x_a^lambda): a_R = i a e^{i tau}, b_R = b e^{2i tau}."""
    za, zb = z[::2], z[1::2]
    x_a = za[0]/np.sqrt(2) + sum(za[2*k-1]*np.sin(k*t) + za[2*k]*np.cos(k*t) for k in range(1, N_H+1))
    p_a = sum(k*za[2*k-1]*np.cos(k*t) - k*za[2*k]*np.sin(k*t) for k in range(1, N_H+1))
    x_b = zb[0]/np.sqrt(2) + sum(zb[2*k-1]*np.sin(k*t) + zb[2*k]*np.cos(k*t) for k in range(1, N_H+1))
    p_b = sum(k/2*zb[2*k-1]*np.cos(k*t) - k/2*zb[2*k]*np.sin(k*t) for k in range(1, N_H+1))
    a_R = (x_a + 1j*p_a)/(np.sqrt(2)*phi_a)*np.exp(1j*t)*1j   # x^lambda + i p^lambda = sqrt2 phi a
    b_R = (x_b + 1j*p_b)/(np.sqrt(2)*phi_b)*np.exp(2j*t)
    return a_R, b_R

In [ ]:
# Seed: integrate M X'' + C X' + K X + f_nl(X) = f_ext(tau) in time at g2/2pi = 100 MHz, from a grid of (x_a, x_a'),
# for all initial conditions at once, then project the last period of each trajectory on the Fourier basis
g2_seed_MHz = 100.0
eps_p_seed = eps_p_of(g2_seed_MHz)
set_params(eps_p_seed)

def rhs(t, y):
    Y = y.reshape(4, -1)
    X, V = Y[:2], Y[2:]
    f = f_nl_time(X)
    acc_a = -X[0] - f[0]
    acc_b = (-X[1] - C[1, 1]*V[1] - f[1] - np.sqrt(2.0)*eps_d_p*np.cos(2*t))/M[1, 1]
    return np.concatenate([V[0], V[1], acc_a, acc_b])

grid = np.linspace(-0.7, 0.7, 9)                 # x_a, x_a' = sqrt2 phi_a (Re a, Im a) up to |a| ~ 4.5
XA, VA = np.meshgrid(grid, grid)
m_traj = XA.size
y0 = np.concatenate([XA.ravel(), np.zeros(m_traj), VA.ravel(), np.zeros(m_traj)])
tau_end = 2*np.pi*1500                           # ~ 1500 memory periods ~ 370 ns ~ 112/kappa_b
N_H_seed = 16
t_last = tau_end - 2*np.pi + np.linspace(0, 2*np.pi, 64*N_H_seed, endpoint=False)
sol_t = solve_ivp(rhs, (0, tau_end), y0, method="DOP853", rtol=1e-9, atol=1e-11,
                  t_eval=np.concatenate([t_last - 2*np.pi, t_last]))
Y = sol_t.y.reshape(4, m_traj, -1)
n_s = len(t_last)
G_last = build_Gamma(t_last, N_H_seed, n, omega, nu)
G_last_pinv = np.linalg.pinv(G_last)

orbits = []
for j in range(m_traj):
    X_prev, X_last = Y[:2, j, :n_s], Y[:2, j, n_s:]
    if np.max(np.abs(X_last - X_prev)) > 1e-4:   # not period 1 (yet)
        continue
    z = G_last_pinv @ X_to_x_tilde(X_last)
    a_R, _ = orbit_complex(z, N_H_seed)
    orbits.append((np.mean(np.abs(a_R)), j, z))
print(f"{len(orbits)} period-1 trajectories out of {m_traj}")
print("distinct mean |a_R|:", np.unique(np.round([o[0] for o in orbits], 2)))

In [ ]:
# The satellite is the period-1 orbit with the largest |a_R|; refine it with the harmonic balance
_, j_sat, z_seed = max(orbits, key=lambda o: o[0])
N_H = 16
L = setup_HB(N_H)
z_sat = powell_hybrid_solve(z_seed, L, build_b_ext(N_H, eps_d_p, nu), Gamma, Gamma_inverse, n, N)
a_R, b_R = orbit_complex(z_sat, N_H)
print("satellite: mean |a_R| =", np.mean(np.abs(a_R)), ", mean |b_R| =", np.mean(np.abs(b_R)))
fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
for o in orbits:
    a_o, b_o = orbit_complex(o[2], N_H_seed)
    axes[0].plot(a_o.real, a_o.imag, color="grey", lw=0.5)
    axes[1].plot(b_o.real, b_o.imag, color="grey", lw=0.5)
axes[0].plot(a_R.real, a_R.imag, "r--"); axes[1].plot(b_R.real, b_R.imag, "r--")
axes[0].set_xlabel(r"$\mathrm{Re}(a)$"); axes[0].set_ylabel(r"$\mathrm{Im}(a)$")
axes[1].set_xlabel(r"$\mathrm{Re}(b)$"); axes[1].set_ylabel(r"$\mathrm{Im}(b)$")
for ax in axes: ax.set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
# Sweep in eps_p from the seed, down to 0.2 and up to 1.1, with continuation and eigenvalue sorting
eps_p_list = np.linspace(0.2, 1.1, 181)
i_seed = np.argmin(np.abs(eps_p_list - eps_p_seed))

def sweep(indices, z_start):
    out = {}
    z_guess, prev_vals, prev_vecs = z_start.copy(), None, None
    eps_prev = eps_p_seed
    for i in indices:
        try:
            for eps in np.linspace(eps_prev, eps_p_list[i], 3)[1:]:   # small sub-steps for the continuation
                set_params(eps)
                z_guess = powell_hybrid_solve(z_guess, L, build_b_ext(N_H, eps_d_p, nu), Gamma, Gamma_inverse, n, N, verbose=False)
            vals, vecs = Floq_coef(z_guess, L, M, C, N_H, n)
            if prev_vals is not None:
                perm = match_eigs(prev_vals, prev_vecs, vals, vecs, T=2*np.pi*nu/omega)
                vals, vecs = vals[perm], vecs[:, perm]
            out[i] = (vals, z_guess.copy())
            prev_vals, prev_vecs, eps_prev = vals, vecs, eps_p_list[i]
        except Exception as e:
            print(f"stopped at eps_p = {eps_p_list[i]:.4f} (g2/2pi = {g2_of(eps_p_list[i])/2/np.pi*1e3:.1f} MHz): {e}")
            break
    return out

res = {**sweep(range(i_seed, -1, -1), z_sat), **sweep(range(i_seed, len(eps_p_list)), z_sat)}
eigvals_array = np.array([res[i][0] if i in res else np.full(2*n, np.nan + 1j*np.nan) for i in range(len(eps_p_list))])
T = 2*np.pi
Poincare_eig = np.exp(eigvals_array*T)
g2_MHz = g2_of(eps_p_list)/2/np.pi*1e3
print("max Liouville residual:", np.nanmax([abs(liouville_residual(ev, M, C)) for ev in eigvals_array]))
print("satellite found for g2/2pi in", np.nanmin(g2_MHz[~np.isnan(eigvals_array[:, 0])]), "-", np.nanmax(g2_MHz[~np.isnan(eigvals_array[:, 0])]), "MHz")
print("max |mu| =", np.nanmax(np.abs(Poincare_eig)), ", min Re mu =", np.nanmin(np.real(Poincare_eig)))

In [ ]:
floquet_data = {
    "description": "Floquet exponents and multipliers of the satellite orbit of the parity-protected cat vs eps_p, from parity_protected_cat/Hill_method_corrected.ipynb",
    "eps_d_over_g2": eps_d_over_g2,
    "g2_definition": "E_J sin(eps_p) phi_a^2 phi_b",
    "kappa_b": kappa_b,
    "N_H": N_H,
    "T": T,
    "eps_p": eps_p_list.tolist(),
    "g2_MHz": g2_MHz.tolist(),
    "exponents_real": np.real(eigvals_array).tolist(),
    "exponents_imag": np.imag(eigvals_array).tolist(),
    "multipliers_real": np.real(Poincare_eig).tolist(),
    "multipliers_imag": np.imag(Poincare_eig).tolist(),
}
with open("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/variants/floquet_parity_protected_12_10_2026.json", "w") as f:
    json.dump(floquet_data, f)

In [ ]:
# Same style as plot_floquet_multipliers.ipynb (Fig. 4)
FONT_SIZE = 13
plt.rcParams.update({'font.size': FONT_SIZE, 'axes.labelsize': FONT_SIZE, 'axes.titlesize': FONT_SIZE,
                     'xtick.labelsize': FONT_SIZE, 'ytick.labelsize': FONT_SIZE, 'legend.fontsize': FONT_SIZE,
                     'figure.figsize': (width_in, width_in/1.3)})
ok = ~np.isnan(eigvals_array[:, 0])
fig, ax = plt.subplots()
for i in range(2*n):
    ax.plot(g2_MHz[ok], np.real(Poincare_eig[ok, i]))
ax.axhline(-1, ls="--", color="r")
ax.axhline(1, ls="--", color="r")
x_min, x_max = g2_MHz[ok].min(), g2_MHz[ok].max()
ax.axvspan(x_min, x_max, color=(70/256, 149/256, 106/256), alpha=0.5, lw=0, zorder=0)
ax.set_xlim(x_min, x_max)
ax.set_xlabel(r"$g_2/2\pi$"+" (MHz)")
ax.set_ylabel("Real part of\nFloquet multipliers")
plt.tight_layout()
plt.savefig("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/variants/floquet_multiplier_parity_protected_12_10_2026.pdf")
plt.show()